# Image Handling

Up to 100 images across all messages can be sent with a single request
- Max file size 5MB
- Height and width of 8000px for one image 
    - 2000px for multiple images 
- Can be done with base64 encoding or URL

- To estimate the token usage, the formula is `(width [px] x height [px])/750`

When a user sends a message with an image, an `ImageBlock` can be appended alongside a `TextBlock` for the assistant message to send back a `TextBlock`

Prompting is still very important when appending an image to the text block.

- Providing guidelines, analysis steps and/or one-shot or multi-shot examples to understand how to approach it
    - If asking it to count the number of items, doing it from specific directions

In [4]:
from utils.util_funcs import *
import base64

client, model = api_client_setup()

In [2]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [6]:
messages = []

with open('./supporting_materials/images/prop7.png', 'rb') as f: 
    image_bytes = base64.standard_b64encode(f.read()).decode('utf-8')

add_user_message(messages, [
        { 
            "type": "image",
            "source": { 
                "type": "base64",
                "media_type": "image/png",
                "data": image_bytes
            }
        },
        { 
            "type": "text", 
            "text": prompt
        }
    ]
)

chat(messages)

Message(id='msg_011CdE8cXekRnYfvyfe2Wswo', container=None, content=[TextBlock(citations=None, text='# Satellite Image Property Fire Risk Analysis\n\n**1. Residence Identification:**\nThe primary residence is a light gray/tan-roofed structure located in the center of the image, appearing as an L-shaped or irregular rectangular building with multiple roof sections, clearly distinguishable from the surrounding dense vegetation.\n\n**2. Tree Overhang Analysis:**\nMultiple trees have canopies that extend directly over significant portions of the residence roof, with estimated coverage of 50-75% of the total roof area, particularly dense on the northern and eastern sections of the structure.\n\n**3. Fire Risk Assessment:**\nThe overhanging trees create multiple ember catch points across the roof surface and establish direct fuel pathways from the surrounding forest canopy to the structure, with branches appearing to be in close proximity or direct contact with the roof in several locations.\